In [11]:
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import json
import warnings

warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, confusion_matrix,
    r2_score, silhouette_score
)
# SECTION 1: DATA LOADING & INSPECTION

print("=" * 70)
print("  SECTION 1: DATA LOADING & INSPECTION")
print("=" * 70)

df = pd.read_csv(r'C:\Users\baps\OneDrive\davproject\digital_diet_mental_health.csv')

print(f"\n Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn overview:")
for col in df.columns:
    dtype = df[col].dtype
    uniq = df[col].nunique()
    nulls = df[col].isnull().sum()
    print(f"  {col:<40} dtype={str(dtype):<10} unique={uniq:<6} nulls={nulls}")

print(f"\nDescriptive Statistics:")
print(df.describe().round(2).to_string())

# SECTION 2: PREPROCESSING

print("\n" + "=" * 70)
print("  SECTION 2: PREPROCESSING")
print("=" * 70)

df_clean = df.copy()

# 2a. Label Encoding for categorical columns
le = LabelEncoder()
df_clean['gender_enc']   = le.fit_transform(df_clean['gender'])
df_clean['location_enc'] = le.fit_transform(df_clean['location_type'])
print("\n Label Encoding applied to: gender, location_type")

# 2b. Age binning
df_clean['age_group'] = pd.cut(
    df_clean['age'],
    bins=[0, 17, 25, 35, 50, 100],
    labels=['Teen(≤17)', 'Young Adult(18-25)', 'Adult(26-35)', 'Middle Age(36-50)', 'Senior(51+)']
)
print(" Age binned into 5 groups")

# 2c. Screen time binning
df_clean['screen_bin'] = pd.cut(
    df_clean['daily_screen_time_hours'],
    bins=[0, 3, 6, 9, 20],
    labels=['Low (0-3h)', 'Medium (3-6h)', 'High (6-9h)', 'Very High (9h+)']
)
print(" Screen time binned into 4 tiers")

# 2d. Mental health category for classification
df_clean['mh_category'] = pd.cut(
    df_clean['mental_health_score'],
    bins=[0, 33, 66, 100],
    labels=['Low', 'Medium', 'High']
)
print("Mental health score → 3-class labels: Low/Medium/High")

# 2e. Feature list for ML
FEATURES = [
    'age', 'daily_screen_time_hours', 'social_media_hours',
    'sleep_duration_hours', 'sleep_quality', 'stress_level',
    'physical_activity_hours_per_week', 'caffeine_intake_mg_per_day',
    'mindfulness_minutes_per_day', 'weekly_anxiety_score',
    'weekly_depression_score', 'mood_rating', 'gaming_hours',
    'entertainment_hours', 'work_related_hours',
    'gender_enc', 'location_enc', 'uses_wellness_apps', 'eats_healthy'
]
TARGET_CLASS = 'mh_category'
TARGET_REG   = 'mental_health_score'

df_ml = df_clean.dropna(subset=[TARGET_CLASS])
X = df_ml[FEATURES]
y_class = df_ml[TARGET_CLASS]
y_reg   = df_ml[TARGET_REG].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f" StandardScaler applied to {len(FEATURES)} features")
print(f" ML dataset: {X.shape[0]} samples × {X.shape[1]} features")

# Train/test split
X_tr, X_te, y_cl_tr, y_cl_te = train_test_split(
    X_scaled, y_class, test_size=0.2, random_state=42, stratify=y_class)
X_tr2, X_te2, y_rg_tr, y_rg_te = train_test_split(
    X_scaled, y_reg, test_size=0.2, random_state=42)
print(f" Train: {X_tr.shape[0]} | Test: {X_te.shape[0]} (80/20 stratified split)")

# SECTION 3: EXPLORATORY DATA ANALYSIS

print("\n" + "=" * 70)
print("  SECTION 3: EXPLORATORY DATA ANALYSIS")
print("=" * 70)

print("\nGender Distribution:")
print(df['gender'].value_counts().to_string())

print("\nLocation Distribution:")
print(df['location_type'].value_counts().to_string())

print("\nAge Group Summary:")
print(df_clean.groupby('age_group')['mental_health_score'].agg(['mean','std','count']).round(2).to_string())

print("\nScreen Time Bin Summary:")
print(df_clean.groupby('screen_bin')['mental_health_score'].agg(['mean','std','count']).round(2).to_string())

print("\nGender vs Key Mental Health Metrics:")
print(df_clean.groupby('gender')[['mental_health_score','stress_level','weekly_anxiety_score','weekly_depression_score']].mean().round(2).to_string())

print("\nLocation vs Key Metrics:")
print(df_clean.groupby('location_type')[['mental_health_score','social_media_hours','stress_level']].mean().round(2).to_string())

# SECTION 4: CORRELATION ANALYSIS

print("\n" + "=" * 70)
print("  SECTION 4: CORRELATION ANALYSIS (Pearson)")
print("=" * 70)

key_cols = [
    'daily_screen_time_hours', 'social_media_hours', 'sleep_duration_hours',
    'sleep_quality', 'mood_rating', 'stress_level',
    'physical_activity_hours_per_week', 'mental_health_score',
    'weekly_anxiety_score', 'weekly_depression_score',
    'caffeine_intake_mg_per_day', 'mindfulness_minutes_per_day'
]
corr = df_clean[key_cols].corr().round(3)
print("\nTop correlations with mental_health_score:")
mh_corr = corr['mental_health_score'].drop('mental_health_score').sort_values(key=abs, ascending=False)
print(mh_corr.to_string())

# SECTION 5: ML ALGORITHM 1 — RANDOM FOREST CLASSIFIER

print("\n" + "=" * 70)
print("  SECTION 5: RANDOM FOREST CLASSIFIER")
print("  Target: Predict Mental Health Category (Low / Medium / High)")
print("=" * 70)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_tr, y_cl_tr)
y_rf_pred = rf.predict(X_te)

print("\nClassification Report:")
print(classification_report(y_cl_te, y_rf_pred))

print("Confusion Matrix:")
cm = confusion_matrix(y_cl_te, y_rf_pred, labels=['Low', 'Medium', 'High'])
print(pd.DataFrame(cm, index=['Act:Low','Act:Med','Act:High'], columns=['Pred:Low','Pred:Med','Pred:High']).to_string())

# Cross-validation
cv_scores = cross_val_score(rf, X_scaled, y_class, cv=5, scoring='accuracy')
print(f"\n5-Fold CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nTop 10 Feature Importances:")
print(feat_imp.head(10).round(4).to_string())

# SECTION 6: ML ALGORITHM 2 — GRADIENT BOOSTING REGRESSOR

print("\n" + "=" * 70)
print("  SECTION 6: GRADIENT BOOSTING REGRESSOR")
print("  Target: Predict continuous Mental Health Score (0-100)")
print("=" * 70)

gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(X_tr2, y_rg_tr)
y_gb_pred = gb.predict(X_te2)

r2   = r2_score(y_rg_te, y_gb_pred)
rmse = np.sqrt(mean_squared_error(y_rg_te, y_gb_pred))
mae  = np.mean(np.abs(y_rg_te - y_gb_pred))

print(f"\nR² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.4f}")
print(f"MAE      : {mae:.4f}")
print(f"\nInterpretation: R² ≈ {r2:.2f} indicates the model explains only ~{max(0,r2)*100:.0f}% of")
print("variance in mental health scores. The target is highly variable relative")
print("to the available digital habit features — suggesting unmeasured confounders.")

# SECTION 7: ML ALGORITHM 3 — K-MEANS CLUSTERING + PCA

print("\n" + "=" * 70)
print("  SECTION 7: K-MEANS CLUSTERING (k=4) + PCA")
print("=" * 70)

# Determine optimal k (Elbow method)
inertias = []
k_range  = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
print("\nElbow method inertias:")
for k, ine in zip(k_range, inertias):
    print(f"  k={k}: {ine:.0f}")

# Final model with k=4
kmeans   = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
sil      = silhouette_score(X_scaled, clusters)
print(f"\nSilhouette Score (k=4): {sil:.4f}")

df_ml2           = df_ml.copy()
df_ml2['cluster'] = clusters

# Cluster profiles
profile_cols = [
    'mental_health_score', 'daily_screen_time_hours', 'stress_level',
    'sleep_quality', 'physical_activity_hours_per_week', 'social_media_hours'
]
cluster_profiles = df_ml2.groupby('cluster')[profile_cols].mean().round(2)
print("\nCluster Profiles:")
print(cluster_profiles.to_string())

# PCA for visualization
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"\nPCA explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}")
print(f"Total explained: {sum(pca.explained_variance_ratio_):.3f}")

# SECTION 8: MATPLOTLIB VISUALIZATIONS

print("\n" + "=" * 70)
print("  SECTION 8: GENERATING MATPLOTLIB VISUALIZATIONS")
print("=" * 70)

plt.style.use('dark_background')
COLORS = ['#38bdf8', '#818cf8', '#34d399', '#fb923c', '#f472b6', '#a78bfa', '#fbbf24']

# Figure 1: EDA Overview
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Digital Diet & Mental Health — EDA Overview', fontsize=16, fontweight='bold', color='white', y=1.01)

# 1a. Gender pie
gender_counts = df['gender'].value_counts()
axes[0,0].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
            colors=COLORS[:3], startangle=90, textprops={'color':'white'})
axes[0,0].set_title('Gender Distribution', color='white')

# 1b. Location bar
loc_counts = df['location_type'].value_counts()
axes[0,1].barh(loc_counts.index, loc_counts.values, color=COLORS[3:6])
axes[0,1].set_title('Location Type', color='white')
axes[0,1].set_xlabel('Count', color='#64748b')

# 1c. MH Score histogram
axes[0,2].hist(df['mental_health_score'], bins=30, color=COLORS[0], alpha=0.85, edgecolor='#0a0e1a')
axes[0,2].axvline(df['mental_health_score'].mean(), color=COLORS[2], linestyle='--', lw=2, label=f'Mean: {df["mental_health_score"].mean():.1f}')
axes[0,2].set_title('Mental Health Score Distribution', color='white')
axes[0,2].set_xlabel('Score', color='#64748b')
axes[0,2].legend()

# 1d. Screen time vs MH
axes[1,0].scatter(df['daily_screen_time_hours'], df['mental_health_score'],
                c=COLORS[1], alpha=0.3, s=15)
axes[1,0].set_xlabel('Daily Screen Time (h)', color='#64748b')
axes[1,0].set_ylabel('Mental Health Score', color='#64748b')
axes[1,0].set_title('Screen Time vs Mental Health', color='white')

# 1e. Sleep quality vs MH
axes[1,1].scatter(df['sleep_quality'], df['mental_health_score'],
                c=COLORS[2], alpha=0.3, s=15)
axes[1,1].set_xlabel('Sleep Quality', color='#64748b')
axes[1,1].set_ylabel('Mental Health Score', color='#64748b')
axes[1,1].set_title('Sleep Quality vs Mental Health', color='white')

# 1f. Stress vs Anxiety
axes[1,2].scatter(df['stress_level'], df['weekly_anxiety_score'],
                c=COLORS[4], alpha=0.3, s=15)
axes[1,2].set_xlabel('Stress Level', color='#64748b')
axes[1,2].set_ylabel('Weekly Anxiety Score', color='#64748b')
axes[1,2].set_title('Stress vs Anxiety', color='white')

for ax in axes.flat:
    ax.tick_params(colors='#64748b')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1e2d4a')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.close()
print("✓ eda_overview.png saved")

#  Figure 2: Correlation Heatmap
fig, ax = plt.subplots(figsize=(13, 10))
mask = np.zeros_like(corr, dtype=bool)
cmap = sns.diverging_palette(340, 200, s=85, l=50, as_cmap=True)
sns.heatmap(corr, ax=ax, cmap=cmap, center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size':8},
            linewidths=0.5, linecolor='#0a0e1a',
            cbar_kws={'shrink':0.8})
ax.set_title('Pearson Correlation Matrix — Key Variables', fontsize=14, color='white', pad=15)
ax.tick_params(colors='#94a3b8', labelsize=9)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.close()
print(" correlation_heatmap.png saved")

#  Figure 3: ML Results
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 3a. Feature importance
ax1 = fig.add_subplot(gs[0, 0])
fi_top10 = feat_imp.head(10)
ax1.barh(fi_top10.index[::-1], fi_top10.values[::-1], color=COLORS[:10])
ax1.set_title('RF Feature Importance (Top 10)', color='white')
ax1.set_xlabel('Importance', color='#64748b')
ax1.tick_params(colors='#94a3b8', labelsize=9)

# 3b. Confusion matrix
ax2 = fig.add_subplot(gs[0, 1])
cm_df = pd.DataFrame(cm, index=['Low','Med','High'], columns=['Low','Med','High'])
sns.heatmap(cm_df, ax=ax2, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, linecolor='#0a0e1a')
ax2.set_title('Confusion Matrix (RF)', color='white')
ax2.set_xlabel('Predicted', color='#64748b'); ax2.set_ylabel('Actual', color='#64748b')

# 3c. Actual vs Predicted (regression)
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(y_rg_te[:200], y_gb_pred[:200], c=COLORS[1], alpha=0.5, s=20)
ax3.plot([0,100],[0,100], '--', color='#64748b', lw=1)
ax3.set_title(f'GB Regressor: Actual vs Predicted\nR²={r2:.3f}, RMSE={rmse:.2f}', color='white')
ax3.set_xlabel('Actual Score', color='#64748b')
ax3.set_ylabel('Predicted Score', color='#64748b')

# 3d. PCA Cluster plot
ax4 = fig.add_subplot(gs[1, :2])
scatter = ax4.scatter(X_pca[:,0], X_pca[:,1], c=clusters, cmap='tab10', alpha=0.6, s=15)
ax4.set_title('K-Means Clusters (k=4) — PCA 2D Projection', color='white')
ax4.set_xlabel('PC1', color='#64748b'); ax4.set_ylabel('PC2', color='#64748b')
handles, labels_ = scatter.legend_elements()
ax4.legend(handles, [f'Cluster {i}' for i in range(4)], loc='upper right', fontsize=9)

# 3e. Cluster profiles radar (bar)
ax5 = fig.add_subplot(gs[1, 2])
x_pos = np.arange(len(profile_cols))
width = 0.2
for ci in range(4):
    vals = cluster_profiles.loc[ci].values
    norm = vals / np.array([100, 15, 10, 10, 10, 6])  # normalize each to 0-1
    ax5.bar(x_pos + ci*width, norm, width, label=f'C{ci}', color=COLORS[ci], alpha=0.85)
ax5.set_xticks(x_pos + 1.5*width)
ax5.set_xticklabels(['MH','Screen','Stress','Sleep','Activity','Social'], fontsize=9, color='#94a3b8')
ax5.set_title('Cluster Profiles (Normalized)', color='white')
ax5.legend(fontsize=9)

for ax in [ax1, ax2, ax3, ax4, ax5]:
    ax.tick_params(colors='#64748b')
    for spine in ax.spines.values(): spine.set_edgecolor('#1e2d4a')

fig.suptitle('ML Results — Random Forest, Gradient Boosting & K-Means Clustering',
            fontsize=14, fontweight='bold', color='white')
plt.savefig('ml_results.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.close()
print(" ml_results.png saved")

# SECTION 9: SUMMARY REPORT

print("\n" + "=" * 70)
print("  SECTION 9: SUMMARY REPORT")
print("=" * 70)
print(f"""
Dataset Summary
───────────────
Total users          : {len(df):,}
Features             : {df.shape[1]}
Age range            : {df['age'].min()} – {df['age'].max()}
Avg daily screen time: {df['daily_screen_time_hours'].mean():.2f} h
Avg mental health    : {df['mental_health_score'].mean():.2f} /100
Avg sleep duration   : {df['sleep_duration_hours'].mean():.2f} h
Avg stress level     : {df['stress_level'].mean():.2f} /10
Avg weekly anxiety   : {df['weekly_anxiety_score'].mean():.2f}

Preprocessing Applied
──────────────────────
Label Encoding    : gender (3 values), location_type (3 values)
StandardScaler    : All 19 numerical features scaled to z-score
Binning           : age → 5 groups, screen time → 4 tiers
Class creation    : mental_health_score → Low/Medium/High (0-33/34-66/67-100)
Train/Test split  : 80% / 20% (stratified for classification)

ML Algorithms
─────────────
Random Forest Classifier
    → Accuracy (test)  : {classification_report(y_cl_te, y_rf_pred, output_dict=True)['accuracy']:.3f}
    → 5-Fold CV Acc    : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}
    → Key finding      : Model biased toward majority "Medium" class
    → Top predictors   : {', '.join(feat_imp.head(3).index.tolist())}

Gradient Boosting Regressor
    → R² Score         : {r2:.4f}
    → RMSE             : {rmse:.4f}
    → Key finding      : Mental health score not well predicted by digital
                        habit features alone (low explained variance)

K-Means Clustering (k=4)
    → Silhouette Score : {sil:.4f}
    → Key finding      : All 4 clusters have near-identical MH score means
                        (48.8–50.8), indicating digital habits don't cleanly
                        separate users by mental health outcome

Correlation Insights
────────────────────
All Pearson correlations between key variables: |r| < 0.06
This indicates weak linear relationships — the associations in this dataset
are either non-linear, mediated by unmeasured factors, or the dataset
was synthetically generated with low correlation design.

Output Files
────────────
eda_overview.png          — EDA charts
correlation_heatmap.png   — Pearson matrix
ml_results.png            — ML algorithm outputs
digital_diet_mental_health_dashboard.html — Interactive D3.js dashboard
""")

print("=" * 70)
print(" Analysis Complete!")
print("=" * 70)









  SECTION 1: DATA LOADING & INSPECTION

 Dataset shape: 2000 rows × 25 columns

Column overview:
  user_id                                  dtype=str        unique=2000   nulls=0
  age                                      dtype=int64      unique=52     nulls=0
  gender                                   dtype=str        unique=3      nulls=0
  daily_screen_time_hours                  dtype=float64    unique=114    nulls=0
  phone_usage_hours                        dtype=float64    unique=73     nulls=0
  laptop_usage_hours                       dtype=float64    unique=52     nulls=0
  tablet_usage_hours                       dtype=float64    unique=26     nulls=0
  tv_usage_hours                           dtype=float64    unique=48     nulls=0
  social_media_hours                       dtype=float64    unique=58     nulls=0
  work_related_hours                       dtype=float64    unique=54     nulls=0
  entertainment_hours                      dtype=float64    unique=63     nulls=0
 